# 面试问题：多 Agent Handoff 和共享 Blackboard 怎样避免串单与越权？

可以直接复述的回答是：第一，Handoff 必须是显式协议，包含 run_id、tenant、来源、目标、允许字段和状态版本。第二，Blackboard 要按运行与租户隔离，不能让多个 Agent 读写一个全局字典。第三，接收方只消费合同允许的字段，并验证前置状态。第四，每次交接都要写入不可变事件账本。第五，重复、迟到和跨租户消息必须被拒绝。第六，用串单率、完成率和交接轨迹评估系统，而不是只看最终回答。下面以企业保险理赔协作为例。

## 真实案例：理赔分诊、保单核验与结算 Agent 协作

五条脱敏理赔请求来自两个企业租户，字段包括 claim_id、policy_id、类型、金额和材料状态。分诊 Agent 识别路线，保单 Agent 核验责任，结算 Agent 给出下一步。数据是与真实理赔流程同构的离线教学样本，不包含姓名、证件或真实保单；结论不能外推为保险业务规则。

In [1]:
claims = [  # 定义五条跨两个租户的脱敏理赔请求
    {"run_id": "RUN-501", "tenant": "north", "claim_id": "CL-501", "policy_id": "P-1001", "category": "门诊", "amount": 860, "documents": "发票+处方"},  # 北区租户的小额门诊申请
    {"run_id": "RUN-502", "tenant": "south", "claim_id": "CL-502", "policy_id": "P-2008", "category": "住院", "amount": 18000, "documents": "发票+出院记录"},  # 南区租户的大额住院申请
    {"run_id": "RUN-503", "tenant": "north", "claim_id": "CL-503", "policy_id": "P-1015", "category": "牙科", "amount": 2300, "documents": "仅发票"},  # 材料不完整的牙科申请
    {"run_id": "RUN-504", "tenant": "south", "claim_id": "CL-504", "policy_id": "P-2012", "category": "门诊", "amount": 420, "documents": "发票+病历"},  # 南区租户的低额门诊申请
    {"run_id": "RUN-505", "tenant": "north", "claim_id": "CL-505", "policy_id": "P-1030", "category": "住院", "amount": 7600, "documents": "发票+出院记录+清单"},  # 材料齐全的住院申请
]  # 结束跨租户理赔输入
print("理赔输入：run | tenant | claim | category | amount | documents")  # 展示多 Agent 系统真正接收的业务字段
for claim in claims:  # 逐条输出五个可读理赔请求
    print(f"{claim['run_id']} | {claim['tenant']:5} | {claim['claim_id']} | {claim['category']:4} | {claim['amount']:5} | {claim['documents']}")  # 显示租户和材料边界


理赔输入：run | tenant | claim | category | amount | documents
RUN-501 | north | CL-501 | 门诊   |   860 | 发票+处方
RUN-502 | south | CL-502 | 住院   | 18000 | 发票+出院记录
RUN-503 | north | CL-503 | 牙科   |  2300 | 仅发票
RUN-504 | south | CL-504 | 门诊   |   420 | 发票+病历
RUN-505 | north | CL-505 | 住院   |  7600 | 发票+出院记录+清单


## Baseline / 基线：多个 Agent 共用一个全局字典

天真实现把“当前理赔”放在全局 Blackboard。两个运行交错时，后写入的 RUN-502 覆盖 RUN-501，保单 Agent 会拿错租户和保单。

In [2]:
shared_blackboard = {}  # 创建没有运行隔离的全局共享状态
shared_blackboard["current_claim"] = claims[0]  # RUN-501 的分诊 Agent 写入当前理赔
shared_blackboard["route"] = "policy_check"  # RUN-501 请求交给保单 Agent
shared_blackboard["current_claim"] = claims[1]  # RUN-502 在前一运行读取前覆盖全局状态
baseline_read = shared_blackboard["current_claim"]  # RUN-501 的保单 Agent 读取到错误理赔
crossed = baseline_read["run_id"] != claims[0]["run_id"]  # 判断是否发生跨运行串单
print("RUN-501 期望读取：", claims[0]["run_id"], claims[0]["policy_id"], claims[0]["tenant"])  # 展示原始运行的正确身份
print("全局 Blackboard 实际读取：", baseline_read["run_id"], baseline_read["policy_id"], baseline_read["tenant"])  # 展示被覆盖后的错误身份
print("是否串单：", crossed)  # 明确输出全局字典的失败语义


RUN-501 期望读取： RUN-501 P-1001 north
全局 Blackboard 实际读取： RUN-502 P-2008 south
是否串单： True


## 核心实现：隔离状态与显式 Handoff 合同

每个 run 拥有独立 Blackboard；Handoff 只携带接收方需要的字段。接收时同时校验租户、运行、状态版本和目标 Agent，再写入事件轨迹。

In [3]:
blackboards = {}  # 建立按 run_id 隔离的运行状态仓库
handoff_ledger = []  # 保存所有交接的可审计事件
def start_run(claim):  # 为单条理赔创建租户隔离的 Blackboard
    blackboards[claim["run_id"]] = {"tenant": claim["tenant"], "version": 1, "stage": "triage", "claim": dict(claim), "coverage": None, "decision": None}  # 复制输入并初始化状态版本
def make_handoff(run_id, sender, receiver, fields):  # 创建带最小字段集的显式交接信封
    board = blackboards[run_id]  # 只读取当前运行自己的隔离状态
    payload = {field: board["claim"].get(field, board.get(field)) for field in fields}  # 按合同白名单挑选接收字段
    return {"handoff_id": f"{run_id}:{board['version']}:{sender}:{receiver}", "run_id": run_id, "tenant": board["tenant"], "version": board["version"], "from": sender, "to": receiver, "payload": payload}  # 返回身份与版本完整的信封
def accept_handoff(envelope, expected_run, expected_tenant, expected_receiver):  # 校验交接身份后推进当前运行
    if envelope["run_id"] != expected_run or envelope["tenant"] != expected_tenant:  # 跨运行或跨租户消息不能进入当前上下文
        return False, "isolation_mismatch"  # 返回明确隔离错误而不泄露载荷
    board = blackboards[expected_run]  # 获取目标运行的隔离 Blackboard
    if envelope["version"] != board["version"]:  # 迟到交接不能覆盖更新后的状态
        return False, "stale_version"  # 返回版本冲突供重放审计
    if envelope["to"] != expected_receiver:  # 其他 Agent 的信封不能被当前接收方消费
        return False, "wrong_receiver"  # 返回接收方合同错误
    handoff_ledger.append((envelope["handoff_id"], envelope["from"], envelope["to"], "accepted"))  # 记录成功交接轨迹
    return True, envelope["payload"]  # 只向接收方暴露白名单载荷
for claim in claims:  # 为五条请求分别创建独立运行
    start_run(claim)  # 初始化当前理赔的版本化 Blackboard
first_envelope = make_handoff("RUN-501", "triage", "policy", ("claim_id", "policy_id", "category", "amount"))  # 为 RUN-501 构造最小保单核验信封
accepted, policy_payload = accept_handoff(first_envelope, "RUN-501", "north", "policy")  # 在目标上下文校验并接收交接
print("Handoff 信封：", first_envelope)  # 展示来源、目标、版本和最小载荷
print("Policy Agent 可见字段：", policy_payload)  # 证明接收方看不到无关材料字段
print("交接轨迹：", handoff_ledger)  # 展示可审计状态迁移


Handoff 信封： {'handoff_id': 'RUN-501:1:triage:policy', 'run_id': 'RUN-501', 'tenant': 'north', 'version': 1, 'from': 'triage', 'to': 'policy', 'payload': {'claim_id': 'CL-501', 'policy_id': 'P-1001', 'category': '门诊', 'amount': 860}}
Policy Agent 可见字段： {'claim_id': 'CL-501', 'policy_id': 'P-1001', 'category': '门诊', 'amount': 860}
交接轨迹： [('RUN-501:1:triage:policy', 'triage', 'policy', 'accepted')]


## 失败案例与修正：跨租户信封不能复用

攻击或编排错误可能把 north 租户的 RUN-501 信封送进 south 租户的 RUN-502。仅检查目标 Agent 会接受它；完整门禁必须先匹配运行与租户，并且拒绝时不回显敏感 payload。

In [4]:
naive_cross_tenant_accept = first_envelope["to"] == "policy"  # 模拟只检查接收方名称的错误门禁
safe_cross_tenant_accept, safe_cross_tenant_reason = accept_handoff(first_envelope, "RUN-502", "south", "policy")  # 在另一个租户上下文尝试消费信封
blackboards["RUN-501"]["version"] = 2  # 模拟 RUN-501 在等待期间已经推进状态版本
stale_accept, stale_reason = accept_handoff(first_envelope, "RUN-501", "north", "policy")  # 尝试重放旧版本交接
print("跨租户修正前：仅检查 receiver，接受=", naive_cross_tenant_accept)  # 展示错误门禁会接收外部租户信封
print("跨租户修正后：接受=", safe_cross_tenant_accept, "原因=", safe_cross_tenant_reason)  # 展示租户与运行隔离拒绝结果
print("迟到 Handoff：接受=", stale_accept, "原因=", stale_reason)  # 展示版本门禁对重放消息的处理


跨租户修正前：仅检查 receiver，接受= True
跨租户修正后：接受= False 原因= isolation_mismatch
迟到 Handoff：接受= False 原因= stale_version


## 结果表：五条理赔的协作路线与隔离指标

In [5]:
coverage_rules = {"门诊": 0.8, "住院": 0.9, "牙科": 0.5}  # 定义教学用保单责任比例
results = []  # 收集五个独立运行的最终协作结果
for claim in claims:  # 逐条模拟分诊、保单核验和结算
    complete_documents = "+" in claim["documents"] and claim["documents"] != "仅发票"  # 用材料字段判断是否可继续核验
    coverage = coverage_rules[claim["category"]] if complete_documents else 0.0  # 材料不完整时不计算赔付
    decision = "request_documents" if not complete_documents else ("manual_review" if claim["amount"] > 10000 else "settle")  # 根据材料和金额生成下一步
    blackboards[claim["run_id"]]["coverage"] = coverage  # 只更新当前运行的责任比例
    blackboards[claim["run_id"]]["decision"] = decision  # 只更新当前运行的结算决定
    results.append((claim["run_id"], claim["tenant"], coverage, decision))  # 保存可展示的最终结果
print("run | tenant | coverage | decision")  # 输出多 Agent 协作后的逐运行结果
for result in results:  # 逐条展示五个隔离运行
    print(" | ".join(map(str, result)))  # 格式化责任比例和下一步动作
isolated_runs = sum(board["claim"]["run_id"] == run_id for run_id, board in blackboards.items())  # 统计 Blackboard 身份一致的运行数
print(f"隔离指标：身份一致={isolated_runs}/{len(blackboards)}，跨租户拒绝=1，已接受交接={len(handoff_ledger)}")  # 汇总隔离和交接健康度


run | tenant | coverage | decision
RUN-501 | north | 0.8 | settle
RUN-502 | south | 0.9 | manual_review
RUN-503 | north | 0.0 | request_documents
RUN-504 | south | 0.8 | settle
RUN-505 | north | 0.9 | settle
隔离指标：身份一致=5/5，跨租户拒绝=1，已接受交接=1


## 结果解读

全局 Blackboard 在两个写入后立刻发生串单；隔离实现让五个 run 的租户和 claim_id 始终一致。Policy Agent 只看见核验所需字段，跨租户与迟到信封分别被 isolation_mismatch 和 stale_version 拒绝。Handoff 的价值不是“把一段文本发给另一个 Agent”，而是建立可验证的最小权限状态转移。

## 生产边界

真实系统需要持久化事件存储、租户级加密密钥、字段级脱敏、消息签名、唯一 handoff_id、并发版本 CAS、超时回收和人工接管。Agent 之间不能共享原始提示词或完整客户档案；最终赔付必须由权威保单与理赔系统回读。本例规则和金额阈值仅用于教学。

## 最小回归测试

In [6]:
assert len(claims) >= 5  # 保证案例至少包含五条跨租户业务请求
assert crossed is True  # 保证全局 Blackboard 基线真实暴露串单
assert accepted is True and set(policy_payload) == {"claim_id", "policy_id", "category", "amount"}  # 保证合法 Handoff 只暴露白名单字段
assert safe_cross_tenant_accept is False and safe_cross_tenant_reason == "isolation_mismatch"  # 保证跨租户信封被隔离门禁拒绝
assert stale_accept is False and stale_reason == "stale_version"  # 保证迟到交接不能覆盖新状态
assert isolated_runs == len(claims)  # 保证五个 Blackboard 的运行身份全部一致
